# Intent classification with PhayaThaiBERT (encoder)

Fine-tunes [`clicknext/phayathaibert`](https://huggingface.co/clicknext/phayathaibert) for **multi-class intent classification** using conversational context + the latest user message.

**Hardware target:** NVIDIA T4 (16 GB VRAM). Full fine-tuning fits comfortably; QLoRA is not used for this encoder baseline.

**Phase 2 alignment:** Use the same JSON file path and `RANDOM_SEED` / split ratios when training a decoder (e.g. Qwen + QLoRA) so metrics are comparable on the same test split.

This notebook has two parts: **training** (sections below) and **inference** (load the saved checkpoint and predict intents).

## Dataset schema and flattening rules

Input JSON is an **array** of objects:

- `user_message` (string): latest user utterance to classify.
- `intent` (string): label, e.g. `MOBILE_BILLING_CHECK_DUE_DATE`.
- `session_history` (array): prior turns, each `{ "role": "user" | "assistant", "content": "..." }`.

BERT has no chat template. We serialize one training example into a **single string**:

1. For each turn in `session_history` **in order**, emit a line `User: ...` or `Assistant: ...` from `role` and `content`.
2. Append a final line `Current user message: ...` using `user_message`.

A future decoder notebook can use the **same JSON** with chat templates; only the string format differs.

### Install dependencies (OpenShift / clean environments)

Installs `transformers`, `datasets`, `evaluate`, and `scikit-learn` if your workbench image does not already include them. Skip or comment out this cell if packages are preinstalled.

In [5]:
!pip install scikit-learn evaluate transformers -q

### Imports and configuration

- Sets **random seeds** for reproducibility (match Phase 2 / Qwen runs with the same seed and split ratios).
- **`MODEL_NAME`**: base Hugging Face checkpoint.
- **`DATA_PATH`**: JSON array file; override with env var `INTENT_DATA_PATH`.
- **`OUTPUT_DIR`**: training logs and saved `final` model; override with `INTENT_OUTPUT_DIR`.
- **`MAX_LENGTH`**: tokenizer truncation length (Thai text can be token-rich).

In [6]:
import json
import os
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

# --- CONFIG: adjust paths for your machine ---
MODEL_NAME = "clicknext/phayathaibert"
DATA_PATH = os.environ.get("INTENT_DATA_PATH", "data/intents.json")
OUTPUT_DIR = Path(os.environ.get("INTENT_OUTPUT_DIR", "outputs/phayathaibert-intent"))
RANDOM_SEED = 42
MAX_LENGTH = 512
TRAIN_FRAC = 0.8
VAL_FRAC_OF_REST = 0.5

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"MODEL_NAME={MODEL_NAME}")
print(f"DATA_PATH={DATA_PATH}")
print(f"OUTPUT_DIR={OUTPUT_DIR}")

MODEL_NAME=clicknext/phayathaibert
DATA_PATH=data/intents.json
OUTPUT_DIR=outputs/phayathaibert-intent


### Serialize each row to a single string (`flatten_session_to_text`)

Encoders take **one text input** per example. This function mirrors the training format used at inference time so behavior stays consistent.

In [7]:
def flatten_session_to_text(row: dict) -> str:
    """Map one JSON object to a single string for encoder tokenization."""
    lines = []
    for turn in row.get("session_history", []) or []:
        role = (turn.get("role") or "user").strip().lower()
        content = (turn.get("content") or "").strip()
        if role == "assistant":
            lines.append(f"Assistant: {content}")
        else:
            lines.append(f"User: {content}")
    current = (row.get("user_message") or "").strip()
    lines.append(f"Current user message: {current}")
    return "\n".join(lines)

### Load JSON, build label maps, and split train / validation / test

- Builds **`label2id` / `id2label`** from sorted unique intents.
- Requires **at least two examples per intent** so stratified splitting is possible.
- **First split:** stratified train vs holdout (`TRAIN_FRAC`).
- **Second split:** validation vs test on the holdout; uses stratification only when every class in the holdout has at least two samples (otherwise falls back to random split for tiny datasets).
- Prints counts and a short **sample** of flattened text.

In [8]:
data_path = Path(DATA_PATH)
if not data_path.is_file():
    raise FileNotFoundError(
        f"Dataset not found: {data_path.resolve()}\n"
        "Set INTENT_DATA_PATH or place your JSON array at data/intents.json"
    )

with open(data_path, "r", encoding="utf-8") as f:
    raw_rows = json.load(f)

if not isinstance(raw_rows, list) or len(raw_rows) == 0:
    raise ValueError("Expected a non-empty JSON array of objects.")

intents = sorted({row["intent"] for row in raw_rows if "intent" in row})
if not intents:
    raise ValueError("No 'intent' labels found in dataset.")

label2id = {name: i for i, name in enumerate(intents)}
id2label = {i: name for name, i in label2id.items()}
num_labels = len(label2id)

from collections import Counter

_intent_counts = Counter(row["intent"] for row in raw_rows)
for name, cnt in _intent_counts.items():
    if cnt < 2:
        raise ValueError(
            f"Stratified split requires at least 2 examples per intent; "
            f"'{name}' has {cnt}. Add more data or duplicate rows for dev."
        )

texts = [flatten_session_to_text(row) for row in raw_rows]
label_ids = [label2id[row["intent"]] for row in raw_rows]

idx = np.arange(len(texts))
train_idx, temp_idx = train_test_split(
    idx,
    test_size=(1.0 - TRAIN_FRAC),
    stratify=label_ids,
    random_state=RANDOM_SEED,
)
labels_temp = [label_ids[i] for i in temp_idx]
ct_temp = Counter(labels_temp)
can_stratify_val = len(temp_idx) >= 2 and all(
    ct_temp.get(lid, 0) >= 2 for lid in set(labels_temp)
)
if can_stratify_val:
    val_idx, test_idx = train_test_split(
        temp_idx,
        test_size=(1.0 - VAL_FRAC_OF_REST),
        stratify=labels_temp,
        random_state=RANDOM_SEED,
    )
else:
    val_idx, test_idx = train_test_split(
        temp_idx,
        test_size=(1.0 - VAL_FRAC_OF_REST),
        random_state=RANDOM_SEED,
    )


def subset(idxs):
    return Dataset.from_dict(
        {"text": [texts[i] for i in idxs], "labels": [label_ids[i] for i in idxs]}
    )


ds = DatasetDict(
    train=subset(train_idx),
    validation=subset(val_idx),
    test=subset(test_idx),
)

print(f"Examples: {len(raw_rows)} | Labels: {num_labels}")
print(f"Train / val / test: {len(ds['train'])} / {len(ds['validation'])} / {len(ds['test'])}")
print("Sample text:\n", ds["train"][0]["text"][:500], "..." if len(ds["train"][0]["text"]) > 500 else "")

Examples: 12 | Labels: 3
Train / val / test: 9 / 1 / 2
Sample text:
 Current user message: ต่อสายพนักงานให้หน่อย 


### Load tokenizer and sequence-classification model

- Loads **`AutoTokenizer`** and **`AutoModelForSequenceClassification`** with `num_labels` and label maps.
- **`ignore_mismatched_sizes=True`**: replaces the pretrained head with a new classifier for your intents.
- Moves the model to **CUDA** if available, else CPU.

In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Device: {device}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CamembertForSequenceClassification LOAD REPORT from: clicknext/phayathaibert
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Device: cuda


### Tokenize datasets and padding collator

- **`tokenize_batch`:** truncation to `MAX_LENGTH`; padding is handled per batch by **`DataCollatorWithPadding`** (not fixed-length padding here).
- Removes the raw `text` column after tokenization.

In [15]:
def tokenize_batch(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )


tokenized = ds.map(tokenize_batch, batched=True, remove_columns=["text"])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/9 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

### Metrics, `TrainingArguments`, and `Trainer`

- **Metrics:** accuracy and **macro F1** on the validation set each epoch.
- **Training:** learning rate `2e-5`, weight decay, FP16 on GPU, save/load best checkpoint by **F1**.
- Adjust `per_device_train_batch_size` if you hit OOM on T4.

In [16]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"],
        "f1": f1_metric.compute(
            predictions=predictions, references=labels, average="macro"
        )["f1"],
    }


training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    seed=RANDOM_SEED,
    logging_steps=25,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

### Run training

Starts fine-tuning. Progress shows training/validation loss and metrics per epoch.

In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,1.150391,0.000000,0.000000
2,No log,0.916016,0.000000,0.000000
3,No log,0.885254,0.000000,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=6, training_loss=0.9665374755859375, metrics={'train_runtime': 81.0892, 'train_samples_per_second': 0.333, 'train_steps_per_second': 0.074, 'total_flos': 1317108768822.0, 'train_loss': 0.9665374755859375, 'epoch': 3.0})

### Evaluate on the held-out test set

Runs **`classification_report`** with **`labels=list(range(num_labels))`** so every intent appears even if the test split has no examples for some classes (avoids sklearn shape errors on small splits).

In [18]:
test_preds = trainer.predict(tokenized["test"])
pred_ids = np.argmax(test_preds.predictions, axis=-1)
true_ids = test_preds.label_ids

target_names = [id2label[i] for i in range(num_labels)]
# labels= fixes sklearn when the test split omits some classes (len(y_unique) < num_labels)
print(
    classification_report(
        true_ids,
        pred_ids,
        labels=list(range(num_labels)),
        target_names=target_names,
        zero_division=0,
    )
)

                               precision    recall  f1-score   support

        GENERAL_CONTACT_AGENT       0.00      0.00      0.00         0
MOBILE_BILLING_CHECK_DUE_DATE       0.50      1.00      0.67         1
          MOBILE_PLAN_INQUIRY       0.00      0.00      0.00         1

                     accuracy                           0.50         2
                    macro avg       0.17      0.33      0.22         2
                 weighted avg       0.25      0.50      0.33         2



### Save model, tokenizer and label map

Writes to **`OUTPUT_DIR / "final"`**: fine-tuned weights, tokenizer, **`label2id.json`**, and **`run_meta.json`** (split settings for reproducibility and Phase 2 comparison).

In [19]:
final_dir = OUTPUT_DIR / "final"
final_dir.mkdir(parents=True, exist_ok=True### Save model, tokenizer, and label map

Writes to **`OUTPUT_DIR / "final"`**: fine-tuned weights, tokenizer, **`label2id.json`**, and **`run_meta.json`** (split settings for reproducibility and Phase 2 comparison).

trainer.save_model(str(final_dir))
tokenizer.save_pretrained(str(final_dir))

label_map_path = final_dir / "label2id.json"
with open(label_map_path, "w", encoding="utf-8") as f:
    json.dump({"label2id": label2id, "id2label": {str(k): v for k, v in id2label.items()}}, f, ensure_ascii=False, indent=2)

meta = {
    "model_name": MODEL_NAME,
    "data_path": str(Path(DATA_PATH).resolve()),
    "random_seed": RANDOM_SEED,
    "train_frac": TRAIN_FRAC,
    "val_frac_of_rest": VAL_FRAC_OF_REST,
    "max_length": MAX_LENGTH,
    "num_train": len(ds["train"]),
    "num_val": len(ds["validation"]),
    "num_test": len(ds["test"]),
}
with open(final_dir / "run_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f"Saved model, tokenizer, label2id, and run_meta under: {final_dir}")
print(
    "Phase 2 (Qwen + QLoRA): use the same DATA_PATH (or copy run_meta.json fields), "
    f"RANDOM_SEED={RANDOM_SEED}, TRAIN_FRAC={TRAIN_FRAC}, VAL_FRAC_OF_REST={VAL_FRAC_OF_REST} "
    "so the train/val/test split matches for comparable metrics."
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model, tokenizer, label2id, and run_meta under: outputs/phayathaibert-intent/final
Phase 2 (Qwen + QLoRA): use the same DATA_PATH (or copy run_meta.json fields), RANDOM_SEED=42, TRAIN_FRAC=0.8, VAL_FRAC_OF_REST=0.5 so the train/val/test split matches for comparable metrics.


---

## Inference

Load the **saved** checkpoint from `OUTPUT_DIR / "final"` (no need to download the base model again for classification). Uses the same **`flatten_session_to_text`** as training so inputs match.

You can run this section **after training** in the same session, or **restart the kernel** and run only: imports + `flatten_session_to_text` + the inference cells below (set `INFERENCE_MODEL_DIR` to your saved path).

### Load tokenizer and model for inference

- **`INFERENCE_MODEL_DIR`**: defaults to `outputs/phayathaibert-intent/final`; override with env `INTENT_MODEL_DIR` or point to any exported folder that contains `config.json`, weights, tokenizer files, and `label2id.json`.
- Restores **`id2label`** from `label2id.json` for human-readable predictions.

In [24]:
INFERENCE_MODEL_DIR = Path(os.environ.get("INTENT_MODEL_DIR", str(OUTPUT_DIR / "final")))

infer_tokenizer = AutoTokenizer.from_pretrained(INFERENCE_MODEL_DIR)
infer_model = AutoModelForSequenceClassification.from_pretrained(INFERENCE_MODEL_DIR)
infer_model.eval()

infer_device = "cuda" if torch.cuda.is_available() else "cpu"
infer_model.to(infer_device)

with open(INFERENCE_MODEL_DIR / "label2id.json", "r", encoding="utf-8") as f:
    _lm = json.load(f)
infer_id2label = {int(k): v for k, v in _lm["id2label"].items()}
infer_num_labels = len(infer_id2label)

print(f"Loaded inference model from: {INFERENCE_MODEL_DIR}")
print(f"Device: {infer_device} | Labels: {infer_num_labels}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loaded inference model from: outputs/phayathaibert-intent/final
Device: cuda | Labels: 3


### `predict_intent`: classify one chat example

Accepts the **same JSON shape** as training (`user_message`, `session_history`). Returns the predicted intent, confidence (softmax at argmax), and per-class scores.

In [25]:
def predict_intent(row: dict, max_length: int = MAX_LENGTH):
    text = flatten_session_to_text(row)
    inputs = infer_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
    )
    inputs = {k: v.to(infer_device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = infer_model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0]
    pred_id = int(torch.argmax(probs))
    return {
        "intent": infer_id2label[pred_id],
        "confidence": float(probs[pred_id]),
        "scores": {infer_id2label[i]: float(probs[i]) for i in range(len(probs))},
        "text_used": text,
    }

### Example predictions

Two sample requests: one with full session history and one minimal. Inspect **`scores`** for debugging ambiguous cases.

In [26]:
example_a = {
    "user_message": "ช่วยเช็กวันครบกำหนดบิลล่าสุดให้หน่อยครับ",
    "session_history": [
        {"role": "assistant", "content": "สวัสดีค่ะ ติดต่อเรื่องบิลหรือค่ะ"},
        {"role": "user", "content": "ใช่ครับ เป็นรายเดือน"},
    ],
}

example_b = {
    "user_message": "ขอคุยกับพนักงานครับ",
    "session_history": [],
}

for i, ex in enumerate([example_a, example_b], 1):
    out = predict_intent(ex)
    print(f"--- Example {i} ---")
    print("predicted:", out["intent"], "| confidence:", round(out["confidence"], 4))
    print("scores:", {k: round(v, 4) for k, v in out["scores"].items()})
    print()

--- Example 1 ---
predicted: MOBILE_BILLING_CHECK_DUE_DATE | confidence: 0.5646
scores: {'GENERAL_CONTACT_AGENT': 0.3084, 'MOBILE_BILLING_CHECK_DUE_DATE': 0.5646, 'MOBILE_PLAN_INQUIRY': 0.127}

--- Example 2 ---
predicted: MOBILE_BILLING_CHECK_DUE_DATE | confidence: 0.4488
scores: {'GENERAL_CONTACT_AGENT': 0.3546, 'MOBILE_BILLING_CHECK_DUE_DATE': 0.4488, 'MOBILE_PLAN_INQUIRY': 0.1966}



## Validate Hugging Face / safetensors checkpoint (PyTorch)
#### Checks that config + weights load, tokenizer works, and a forward pass returns logits of the expected shape.

In [33]:
from pathlib import Path
import json
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_DIR = Path("outputs/phayathaibert-intent/final")  # or your path

assert (MODEL_DIR / "config.json").is_file(), "missing config.json"
assert (MODEL_DIR / "label2id.json").is_file(), "missing label2id.json"
# weights: safetensors and/or bin
assert (MODEL_DIR / "model.safetensors").is_file() or (MODEL_DIR / "pytorch_model.bin").is_file(), "no weight file"

with open(MODEL_DIR / "label2id.json", "r", encoding="utf-8") as f:
    maps = json.load(f)
label2id = maps["label2id"]
num_labels = len(label2id)
print("num_labels:", num_labels, "labels:", list(label2id.keys()))

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.eval()

text = "User: hello\nCurrent user message: test"
inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
with torch.no_grad():
    out = model(**inputs)
logits = out.logits
print("logits shape:", tuple(logits.shape))  # expect (1, num_labels)
assert logits.shape[-1] == num_labels, "logits dim != num_labels"
print("OK: HF checkpoint loads and forward pass matches num_labels")

num_labels: 3 labels: ['GENERAL_CONTACT_AGENT', 'MOBILE_BILLING_CHECK_DUE_DATE', 'MOBILE_PLAN_INQUIRY']


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

logits shape: (1, 3)
OK: HF checkpoint loads and forward pass matches num_labels


In [58]:
import os
from pathlib import Path
import boto3

MODEL_DIR = Path(os.environ.get(
    "INTENT_MODEL_DIR",
    "./outputs/phayathaibert-intent/final"
))

S3_BUCKET = os.environ.get("S3_BUCKET", "trained-model")
S3_PREFIX = os.environ.get("S3_PREFIX", "phayathaibert-intent").strip("/") + "/"

print(f"MODEL_DIR={MODEL_DIR.resolve()}")
print(f"S3_BUCKET={S3_BUCKET}  S3_PREFIX={S3_PREFIX}")

if not MODEL_DIR.exists():
    raise ValueError(f"Model directory not found: {MODEL_DIR}")

s3 = boto3.client(
    "s3",
    endpoint_url=os.getenv("AWS_S3_ENDPOINT"),
    aws_access_key_id=os.environ.get("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.environ.get("AWS_SECRET_ACCESS_KEY"),
)

for path in MODEL_DIR.rglob("*"):
    if path.is_file():
        key = S3_PREFIX + str(path.relative_to(MODEL_DIR)).replace("\\", "/")
        s3.upload_file(str(path), S3_BUCKET, key)
        print(f"uploaded s3://{S3_BUCKET}/{key}")

print(f"Done. s3://{S3_BUCKET}/{S3_PREFIX}")

MODEL_DIR=/opt/app-root/src/outputs/phayathaibert-intent/final
S3_BUCKET=trained-model  S3_PREFIX=phayathaibert-intent/
uploaded s3://trained-model/phayathaibert-intent/model.safetensors
uploaded s3://trained-model/phayathaibert-intent/run_meta.json
uploaded s3://trained-model/phayathaibert-intent/label2id.json
uploaded s3://trained-model/phayathaibert-intent/tokenizer_config.json
uploaded s3://trained-model/phayathaibert-intent/config.json
uploaded s3://trained-model/phayathaibert-intent/tokenizer.json
uploaded s3://trained-model/phayathaibert-intent/training_args.bin
Done. s3://trained-model/phayathaibert-intent/
